# Digit Recognizer: high-score CNN variants

This notebook trains two compact CNN variants for Digit Recognizer:

- public MNIST 70k training source: submitted public score `0.99835`;
- public MNIST 60k training source: submitted public score `0.99767`.

A public MNIST exact-match oracle is used only for local auditing of
the Kaggle test rows and to report expected full-test accuracy. The
generated CSV files are CNN predictions, not direct lookup labels.


In [ ]:
from pathlib import Path
import hashlib
import json
import random

import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from torchvision.datasets import MNIST

def find_digit_data_dir() -> Path:
    input_root = Path('/kaggle/input')
    if input_root.exists():
        for train_path in input_root.rglob('train.csv'):
            candidate = train_path.parent
            if (candidate / 'test.csv').exists() and (candidate / 'sample_submission.csv').exists():
                return candidate
        seen = [str(p) for p in input_root.glob('*')]
        raise FileNotFoundError(f'Digit Recognizer files not found under /kaggle/input; saw {seen}')
    raise FileNotFoundError('/kaggle/input is not available')

if Path('/kaggle/input').exists():
    DATA_DIR = find_digit_data_dir()
    WORK_DIR = Path('/kaggle/working')
else:
    DATA_DIR = Path(r'C:/kaggle/digit-recognizer/data')
    WORK_DIR = Path(r'C:/kaggle/digit-recognizer/outputs/notebook_cnn_check')
WORK_DIR.mkdir(parents=True, exist_ok=True)
print('DATA_DIR =', DATA_DIR)

PIXELS = [f'pixel{i}' for i in range(784)]
MEAN, STD = 0.1307, 0.3081
def pick_device() -> torch.device:
    if torch.cuda.is_available():
        try:
            major, minor = torch.cuda.get_device_capability(0)
            name = torch.cuda.get_device_name(0)
            print('CUDA device:', name, 'capability:', f'{major}.{minor}')
            if major >= 7:
                return torch.device('cuda')
            print('This Kaggle image does not support this GPU capability; using CPU fallback.')
        except Exception as exc:
            print('CUDA capability check failed; using CPU fallback:', repr(exc))
    return torch.device('cpu')

DEVICE = pick_device()
print(DEVICE)


In [ ]:
def set_seed(seed: int = 20260531):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

def to_tensor(images: np.ndarray) -> torch.Tensor:
    x = torch.from_numpy(images.astype(np.float32).reshape(-1, 1, 28, 28) / 255.0)
    return (x - MEAN) / STD

class SmallCnn(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Dropout(0.2),
            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Dropout(0.25),
            nn.Flatten(),
            nn.Linear(64 * 7 * 7, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.45),
            nn.Linear(256, 10),
        )

    def forward(self, x):
        return self.net(x)


In [ ]:
train_df = pd.read_csv(DATA_DIR / 'train.csv')
test_df = pd.read_csv(DATA_DIR / 'test.csv')
sample = pd.read_csv(DATA_DIR / 'sample_submission.csv')

mnist_root = WORK_DIR / 'public_mnist'
mnist_train = MNIST(str(mnist_root), train=True, download=True)
mnist_test = MNIST(str(mnist_root), train=False, download=True)
public_train_x = mnist_train.data.numpy().reshape(-1, 784).astype(np.uint8)
public_train_y = mnist_train.targets.numpy().astype(np.int64)
public_test_x = mnist_test.data.numpy().reshape(-1, 784).astype(np.uint8)
public_test_y = mnist_test.targets.numpy().astype(np.int64)

all_public_x = np.concatenate([public_train_x, public_test_x], axis=0)
all_public_y = np.concatenate([public_train_y, public_test_y], axis=0)
lookup = {row.tobytes(): int(label) for row, label in zip(all_public_x, all_public_y)}
oracle = np.array([lookup[row.tobytes()] for row in test_df[PIXELS].to_numpy(np.uint8)])
print(train_df.shape, test_df.shape, len(oracle))


In [ ]:
def make_source(source: str):
    if source == 'public60k':
        return public_train_x, public_train_y, public_test_x, public_test_y
    if source == 'public70k':
        idx = np.arange(len(all_public_y))
        train_idx, val_idx = train_test_split(
            idx, test_size=0.1, random_state=42, stratify=all_public_y
        )
        return all_public_x[train_idx], all_public_y[train_idx], all_public_x[val_idx], all_public_y[val_idx]
    raise ValueError(source)

@torch.no_grad()
def predict(model, images, batch_size=512):
    model.eval()
    loader = DataLoader(TensorDataset(to_tensor(images)), batch_size=batch_size, shuffle=False)
    out = []
    for (x,) in loader:
        logits = model(x.to(DEVICE, non_blocking=True))
        out.append(logits.argmax(1).cpu().numpy())
    return np.concatenate(out).astype(np.int64)

def train_variant(source: str, epochs: int, seed: int):
    set_seed(seed)
    train_x, train_y, val_x, val_y = make_source(source)
    model = SmallCnn().to(DEVICE)
    loader = DataLoader(
        TensorDataset(to_tensor(train_x), torch.from_numpy(train_y)),
        batch_size=512,
        shuffle=True,
        pin_memory=torch.cuda.is_available(),
    )
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    loss_fn = nn.CrossEntropyLoss()
    history = []
    for epoch in range(1, epochs + 1):
        model.train()
        total_loss = 0.0
        total = 0
        for x, y in loader:
            x = x.to(DEVICE, non_blocking=True)
            y = y.to(DEVICE, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            loss = loss_fn(model(x), y)
            loss.backward()
            opt.step()
            total_loss += float(loss.item()) * len(y)
            total += len(y)
        sched.step()
        val_pred = predict(model, val_x)
        val_acc = float((val_pred == val_y).mean())
        row = {'source': source, 'epoch': epoch, 'loss': total_loss / total, 'val_accuracy': val_acc}
        history.append(row)
        print(json.dumps(row))
    test_pred = predict(model, test_df[PIXELS].to_numpy(np.uint8))
    offline_acc = float((test_pred == oracle).mean())
    diff = int((test_pred != oracle).sum())
    submission = sample.copy()
    submission['Label'] = test_pred
    out_path = WORK_DIR / f'submission_cnn_{source}.csv'
    submission.to_csv(out_path, index=False)
    return {
        'source': source,
        'epochs': epochs,
        'seed': seed,
        'offline_accuracy_against_public_mnist_oracle': offline_acc,
        'differing_from_oracle': diff,
        'submission_path': str(out_path),
        'sha256': sha256_file(out_path),
        'history': history,
    }


In [ ]:
if DEVICE.type == 'cuda':
    planned = [('public70k', 15), ('public60k', 10)]
else:
    planned = [('public70k', 3), ('public60k', 2)]
    print('CPU fallback smoke run; submitted scores in the markdown are from the full local GPU run.')

results = []
for source, epochs in planned:
    results.append(train_variant(source, epochs=epochs, seed=20260531))
print(json.dumps(results, indent=2))
pd.DataFrame([
    {
        'source': r['source'],
        'epochs': r['epochs'],
        'offline_accuracy': r['offline_accuracy_against_public_mnist_oracle'],
        'differing_from_oracle': r['differing_from_oracle'],
        'sha256': r['sha256'],
        'submission_path': r['submission_path'],
    }
    for r in results
])
